[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/solutions/40_adam_solution.ipynb)

# 🟡 Solution: Adam Optimizer

*Training · Medium*

Reference implementation. Try it yourself in `40_adam.ipynb` first.

---
Implement the **Adam** update as a pure function over pytrees.

$$m_t = \beta_1 m_{t-1} + (1-\beta_1) g_t \qquad
  v_t = \beta_2 v_{t-1} + (1-\beta_2) g_t^2$$

$$\hat{m}_t = \frac{m_t}{1-\beta_1^t} \qquad
  \hat{v}_t = \frac{v_t}{1-\beta_2^t}$$

$$\theta_t = \theta_{t-1} - \eta \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon}$$

### Signature
```python
def adam_update(params, grads, state, step, lr=1e-3,
                b1=0.9, b2=0.999, eps=1e-8):
    # state: {"m": pytree_like_params, "v": pytree_like_params}
    # step:  1-based — the first call has step=1
    # returns (new_params, new_state)
```

### Rules
- Do **not** use `optax`
- `params`, `grads`, `m` and `v` are all the same pytree structure — use
  `jax.tree.map`, do not assume a flat array
- `state` starts as all-zeros `m` and `v`
- Must work under `jax.jit`

### What bias correction actually fixes
$m$ and $v$ both start at **zero**, so both are biased toward zero early on: at
$t=1$, $m_1 = (1-\beta_1) g_1 = 0.1 g_1$ and $v_1 = (1-\beta_2) g_1^2 =
0.001 g_1^2$.

The trap is guessing which way the error goes. "Both estimates are too small, so
the step must be too small" is the intuitive answer and it is **wrong**. The
step is a *ratio*, and $v$ is far more biased than $m$ — it also sits under a
square root, which halves its bias in log terms. Uncorrected, with a constant
gradient:

$$\frac{m_t}{\sqrt{v_t}} = \frac{1-\beta_1^t}{\sqrt{1-\beta_2^t}}
\quad\Longrightarrow\quad
3.2\times \text{ at } t=1, \;\; 6.5\times \text{ at } t=10, \;\;
1.26\times \text{ at } t=1000$$

So an Adam missing bias correction **overshoots by a factor of 3–6 for the first
few hundred steps**, and does not settle within 10% of the right scale until
around $t \approx 1750$ at $\beta_2 = 0.999$. That is a diverging-loss bug, not a
slow-start bug — and it is the same failure mode warmup exists to paper over
([[cosine_lr]]).

The correction has a sharp observable signature: **with** it, the very first
update has magnitude exactly $\eta$ regardless of how large or small the
gradient is (since $\hat{m}_1/\sqrt{\hat{v}_1} = g_1/|g_1| = \pm 1$). That
property is what the tests check, and it is the cleanest way to tell a correct
Adam from one missing the correction.

### Where AdamW differs
AdamW does **not** add weight decay to the gradient. It applies
$\theta \mathrel{-}= \eta \lambda \theta$ separately, so the decay is not scaled
by $\sqrt{\hat{v}}$. Folding L2 into the gradient instead — plain "Adam + L2" —
decays large-gradient parameters *less*, which is why AdamW generalises better
and is the default for transformers.

In [ ]:
# Install jax-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q jax-judge flax')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✅ REFERENCE SOLUTION

import jax
import jax.numpy as jnp


def adam_update(params, grads, state, step, lr=1e-3, b1=0.9, b2=0.999, eps=1e-8):
    m = jax.tree.map(lambda m_, g: b1 * m_ + (1 - b1) * g, state["m"], grads)
    v = jax.tree.map(lambda v_, g: b2 * v_ + (1 - b2) * g * g, state["v"], grads)

    # step is 1-based, so at the first call the denominators are (1 - b1) and
    # (1 - b2) — exactly cancelling the shrinkage of the zero-initialised means.
    mc = 1 - b1 ** step
    vc = 1 - b2 ** step

    new_params = jax.tree.map(
        lambda p, m_, v_: p - lr * (m_ / mc) / (jnp.sqrt(v_ / vc) + eps),
        params, m, v,
    )
    return new_params, {"m": m, "v": v}

In [ ]:
# 🔍 Verify
import jax
import jax.numpy as jnp

params = {"w": jnp.array([1.0, -2.0])}
state = {"m": jax.tree.map(jnp.zeros_like, params),
         "v": jax.tree.map(jnp.zeros_like, params)}

# Wildly different gradient magnitudes...
for g in (jnp.array([1e-4, 1e-4]), jnp.array([1e4, 1e4])):
    p, _ = adam_update(params, {"w": g}, state, step=1, lr=0.1)
    print(f"grad {g[0]:>8.0e} -> step size {abs(float(p['w'][0] - 1.0)):.4f}")
# ...both give a first step of exactly lr. That is bias correction doing its job.

# And the direction of the bug, which is the part people get backwards:
# uncorrected Adam OVERSHOOTS, it does not undershoot.
b1, b2 = 0.9, 0.999
print("
  t   uncorrected |step| / lr")
for t in (1, 10, 100, 1000):
    ratio = (1 - b1 ** t) / jnp.sqrt(1 - b2 ** t)   # constant-gradient case
    print(f"{t:>5}   {float(ratio):.2f}x")

In [ ]:
# Run the judge against the reference solution
from jax_judge import check

check("adam")